# Getting Started with AkasicDB

In this tutorial, you will use AkasicDB and open-source, text-based embeddings, and generative AI models to create a RAG system.

## Prerequisite

* Modern Web Browser, such as Chrome, Firefox, Safari, and etc.
* Google Account (for Google Colab)
* AkasicDB Playground DB instance (You can create one [here](https://db.akasic.cloud).)
* Hugging Face Access Token (To download open-source models. You can create yours [here](https://huggingface.co/settings/tokens).)


## Retrieval-Augmented Generation (RAG) using AkasicDB

Retrieval-augmented generation (RAG) is a technique used with large language models (LLMs) to connect the model with a knowledge base of information outside the data the LLM has been trained on without having to perform fine-tuning. Traditional RAG is limited to text-based use cases such as text summarization and chatbots.

Multimodal RAG can use multimodal LLMs (MLLM) to process information from multiple types of data to be included as part of the external knowledge base used in RAG. Multimodal data can include text, images, audio, video or other forms. Popular multimodal LLMs include Google's Gemini, Meta's Llama 3.2 and OpenAI's GPT-4 and GPT-4o.

In both Traditional RAG and Multimodal RAG, Vector Databases play a crucial role. They are essential for efficiently storing, indexing, and querying high-dimensional vector embeddings of data, which allows for fast and accurate retrieval of relevant information to augment LLM responses.

## Traditional, Text-based RAG

First, we build a simple text-based RAG system using AkasicDB. We will build a multi-modal RAG in the next tutorial.

## Step 0: GPU, HF Access Token, AkasicDB instance

### GPU

We need to check if we have a GPU in the Colab environment.


In [ ]:
!nvidia-smi

### Hugging Face Access Token

Hugging Face is a platform for ML models and datasets. We will use models from Hugging Face so we need an access token.

If you haven't, you can create a new access token [here](https://huggingface.co/settings/tokens). For **Token type**, please select **Read**.

To store and use the access token securely, we use Google Colab's **Secrets**. Open your Google Colab notebook and click the key icon in the left-hand sidebar. Let's name it as `HF_TOKEN`.

### AkasicDB Instance

If you haven't done yet, please go to https://db.akasic.cloud and create a new DB instance. Copy the connection string and also create a Google Colab secret with it. Let's name it as `DB_DSN`.

## Step 1: Install dependencies


In [ ]:
%pip install --quiet requests beautifulsoup4 transformers accelerate huggingface_hub akasicdb

Import required dependencies.

In [ ]:
import os
import textwrap
from typing import Iterable

import argparse
import requests
from bs4 import BeautifulSoup

import numpy as np
import psycopg

from akasicdb.psycopg import register_vector
from akasicdb.types import Hnsw, IndexOption, SparseVector, Vamana
from google.colab import userdata


In [ ]:
# Configure AkasicDB Connection String
DB_DSN = os.getenv("DATABASE_URL", userdata.get("DB_DSN"))
def get_conn(conn_string: str=DB_DSN):
    return psycopg.connect(conn_string)

Load a text embedding model. For now, let's use `Qwen3-Embedding-0.6B` model.

In [ ]:
# Imports for local LLM
from transformers import pipeline, AutoModel, AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F

from huggingface_hub import login

# Login to Hugging Face to download model weights
login(token=userdata.get('HF_TOKEN'))

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


def last_token_pool(last_hidden_states: torch.Tensor,
                    attention_mask: torch.Tensor) -> torch.Tensor:
    # If left padding is used, the last position is the last actual token.
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    # If right padding is used, find and extract the last valid token index for each sequence.
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths,
    ]


@torch.no_grad()
def embed_text(text: str) -> list[float]:
    batch = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=8192,
        return_tensors="pt",
    ).to(device)

    outputs = model(**batch)
    embeddings = last_token_pool(outputs.last_hidden_state, batch["attention_mask"])
    embeddings = F.normalize(embeddings, p=2, dim=1)  # for cosine similarity
    return embeddings[0].cpu().tolist()

@torch.no_grad()
def embed_texts(texts: list[str], batch_size: int = 32) -> list[list[float]]:
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=8192,
            return_tensors="pt",
        ).to(device)
        outputs = model(**batch)
        embeddings = last_token_pool(outputs.last_hidden_state, batch["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        results.extend(embeddings.cpu().tolist())
    return results


This function is responsible for fetching the content of a web page from a given URL. It uses the `requests` library to make an HTTP GET request to the specified URL. To avoid being blocked by some websites, it includes a `User-Agent` header. After successfully fetching the page, it uses `BeautifulSoup` to parse the HTML content. Importantly, it removes `<script>`, `<style>`, and `<noscript>` tags, as these usually contain code or styling information that is not relevant for text-based RAG. Finally, it extracts all the visible text from the HTML, joins it into a single string, and returns it. This process ensures that only the meaningful text content is retrieved from the web page.

In [ ]:
def fetch_blog_text(url: str) -> str:
    """
    Fetch text from the given URL.

    Args:
        url (str): website URL to fetch.

    Returns:
        str: fetched text.
    """
    response = requests.get(
        url,
        timeout=10,
        headers={"User-Agent": "Mozilla/5.0 (rag-example)"},
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return " ".join(text.split())

### Chunk text

This function takes a given `text` and breaks it down into smaller, manageable chunks. The primary parameters are `chunk_size`, which determines the maximum length of each chunk, and `chunk_overlap`, which specifies how many characters should overlap between consecutive chunks. The overlap helps maintain context across chunks, which is crucial for RAG systems. The function iterates through the text, creating chunks of the specified `chunk_size`, and then advances the starting point by `chunk_size - chunk_overlap` for the next chunk. It also ensures that only non-empty chunks are included in the final list. This process is vital for preparing long documents for embedding, as most embedding models have input token limits.

In [ ]:
def chunk_text(text: str, chunk_size: int = 800, chunk_overlap: int = 100) -> list[str]:
    """
    Chunk fetched text.

    Args:
        text (str): Text to be chunked.
        chunk_size (int): chunk size.
        chunk_overlap (int): number of characters to be overlapped.

    Returns:
        list[str]: list of chunked text.
    """
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks


In [ ]:
# -----------------------
# Store chunks and their embeddings
# -----------------------

def upsert_documents(docs: list[str], embeddings: list[list[float]], source: str):
    """
    Store text chunks and their embeddings into the VDB

    Args:
        docs (list[str]): List of text chunks to be stored.
    """
    rows = []
    for i, doc in enumerate(docs):
        rows.append((doc, embeddings[i]))

    sql = """
    ALTER ROLE postgres
    SET search_path TO "$user", public, extensions, vectoron, graphon, akasicdb
    """

    with psycopg.connect(DB_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()

    with psycopg.connect(DB_DSN) as conn:
        conn.execute("CREATE EXTENSION IF NOT EXISTS akasicdb;")
        conn.execute("SELECT akasicdb_admin.initialize();")
        register_vector(conn)
        conn.execute("CREATE TABLE IF NOT EXISTS documents \
            (id SERIAL PRIMARY KEY, content TEXT NOT NULL, embedding vectoron.vector(1024), src TEXT NOT NULL)")
        conn.commit()
        try:
            for doc, embedding in rows:
                conn.execute(
                    "INSERT INTO documents (content, embedding, src) VALUES (%s, %s, %s);",
                    (doc, embedding, source),
                )
        finally:
            conn.commit()


### Search text chunks relevant to the query

The following function `search_similar()` searches text chunks in the vector database similar to the given query. This function performs the following steps:

1.  **Embed Query**: It first uses the `embed_text()` function to convert the input `query` into a high-dimensional vector embedding. This embedding represents the semantic meaning of the query.
2.  **Format Query Embedding**: The vector embedding is then converted into a string literal format suitable for database queries.
3.  **Connect to Database**: It establishes a connection to the AkasicDB instance.
4.  **Execute Similarity Search**: Within a database cursor, it executes a SQL query to select document `id`, `content`, and `src` from the `documents` table. The `ORDER BY embedding <-> %s::vectoron.vector` clause performs a similarity search, ordering results by their vector distance to the `query_embedding`. The `<->` operator is typically used for cosine distance in vector databases.
5.  **Limit Results**: The search is limited to `top_k` results, retrieving the most similar document chunks.
6.  **Format Output**: The fetched results are formatted into a markdown string for display, showing the query, the number of retrieved chunks, and details for each chunk including its ID, source, and a preview of its content within a collapsible `<details>` tag.
7.  **Return Chunks**: Finally, it returns a list containing only the content of the retrieved document chunks, which can then be used as context for an LLM.

In [ ]:
# -----------------------
# Search relevant chunks
# -----------------------

from IPython.display import Markdown, display


def search_similar(query: str, top_k: int = 3) -> list[str]:
    """

    Args:
        query (str):
        top_k (int):

    Returns:
        list[str]:
    """
    query_embedding = embed_text(query)
    query_embedding_literal = "[" + ",".join(str(x) for x in query_embedding) + "]"

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT id, content, src
                FROM documents
                ORDER BY embedding <-> %s::vectoron.vector
                LIMIT %s
                """,
                (query_embedding_literal, top_k)
            )
            results = cur.fetchall()

    md_parts = [f"### 🔍 Query\n\n> {query}\n\n### 📚 Top-{len(results)} retrieved chunks\n"]
    for rank, (doc_id, content, src) in enumerate(results, start=1):
        md_parts.append(
            f"**{rank}. id=`{doc_id}`** · source: <{src}>\n\n"
            f"<details><summary>chunk preview</summary>\n\n"
            f"{content}\n\n"
            f"</details>\n"
        )
    display(Markdown("\n".join(md_parts)))

    return [row[1] for row in results]

In [ ]:

def ingest(url: str, chunk_size: int, chunk_overlap: int):
    fetched_text = fetch_blog_text(url)
    chunks = chunk_text(fetched_text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    embeddings = embed_texts(chunks)
    upsert_documents(chunks, embeddings, url)


In [ ]:
URLS = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
#    "https://aws.amazon.com/ko/what-is/ai-agents/",
]

for url in URLS:
    ingest(url=url, chunk_size=1000, chunk_overlap=200)

# ingest(url="https://lilianweng.github.io/posts/2023-06-23-agent/", chunk_size=1000, chunk_overlap=200)


We have "ingested" text from then blog. Now it's time for use to load a chat model.

In [ ]:
# Define the local chat model
CHAT_MODEL_NAME = "Qwen/Qwen3-1.7B"

# Load tokenizer and model for local chat
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_NAME)
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
)

# Create a text generation pipeline
generator = pipeline(
    "text-generation",
    model=chat_model,
    tokenizer=chat_tokenizer,
    max_new_tokens=500, # Limit response length
    do_sample=True,
    temperature=0.7,
    top_k=20,
    top_p=0.8,
    return_full_text=False,
)


And define some helper functions to generate answer texts. First, `generate_answer()` is a function to generate an answer using a given prompt.

In [ ]:
import json

def parse_llm_response(response_text: str) -> dict:
    """
    Parse LLM response and return type and content.

    Returns:
        dict: {"type": "answer | "error", "content": str}
    """
    stripped = response_text.strip()

    # JSON?
    if stripped.startswith("{"):
        try:
            parsed = json.loads(stripped)
            if "error" in parsed:
                return {"type": "error", "content": parsed["error"]}
        except json.JSONDecodeError:
            pass
    return {"type": "answer", "content": stripped}

# Generate an answer text using LLM
def generate_answer(prompt: str) -> str:
    """
    Return a string containing the generated answer using local LLM.

    Args:
        prompt (str): Prompt.

    Returns:
        str: Generated answer.
    """
    # Chat template
    messages = [
        {"role": "user", "content": prompt}
    ]
    formatted_prompt = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False)

    outputs = generator(formatted_prompt)
    # Extract the generated text, excluding the input prompt
    generated_text = outputs[0]["generated_text"].strip()

    # Remove the input prompt from the generated text
    response_text = generated_text.replace(formatted_prompt, "").strip()
    return parse_llm_response(response_text)


The `rag_answer()` function orchestrates the Retrieval-Augmented Generation (RAG) pipeline for a given query. Here's a breakdown of its role:

- **Retrieval**: It first uses the `search_similar()` function to find the top k most relevant document chunks from the AkasicDB instance based on the input query.
- **Context Construction**: These retrieved document chunks are then combined to form a context that provides relevant information.
- **Prompt Engineering**: A prompt is constructed by integrating the original query and the context. This prompt instructs the large language model (LLM) to answer the question using only the provided context.
- **Generation**: Finally, it calls the `generate_answer()` function (which utilizes the loaded chat model) to generate a response based on the engineered prompt. The result is the RAG-enhanced answer to the user's query.

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant that answers questions strictly based on the provided context.

You MUST follow these rules:
1. Answer ONLY using information explicitly stated in the provided context.
2. Do NOT use any external knowledge or make assumptions beyond the context.
3. You MUST respond in the following JSON format only, with no other text outside the JSON.

If the context contains relevant information to answer the question:
{
  "status": "answered",
  "content": "<answer in markdown format>"
}

If the context does not contain sufficient information to answer the question:
{
  "status": "not_found",
  "content": "The provided context does not contain enough information to answer this question."
}"""


def build_prompt(context: str, query: str) -> list[dict]:
    """
    Returns a messages list for chat template.
    """
    user_message = f"""[Context]
    {context}

    [Question]
    {query}"""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

def parse_response(response_text: str) -> dict:
    # Some models may wrap the response in a ```json ... ``` block.
    cleaned = response_text.strip().removeprefix("```json").removesuffix("```").strip()

    try:
        parsed = json.loads(cleaned)
        status = parsed.get("status")
        content = parsed.get("content", "")

        if status == "answered":
            return {"type": "answer", "content": content}
        elif status == "not_found":
            return {"type": "error", "content": content}
        else:
            return {"type": "error", "content": f"Unexpected response: {response_text}"}

    except json.JSONDecodeError:
        # If JSON parsing fails, fall back to the raw response as the answer.
        return {"type": "answer", "content": response_text}

def generate_answer(context: str, query: str) -> dict:
    messages = build_prompt(context, query)

    formatted_prompt = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    outputs = generator(formatted_prompt)
    generated_text = outputs[0]["generated_text"].strip()
    response_text = generated_text.replace(formatted_prompt, "").strip()

    return parse_response(response_text)

def render_as_markdown(result: dict) -> str:
    if result["type"] == "error":
        return f"> ⚠️ **I can't answer the question based on the retrieved context**\n>\n> {result['content']}"
    return result["content"]

In [ ]:
def rag_answer(query: str) -> str:
    """
    Generate an answer to a given query using RAG pipeline.

    Args:
        query (str): query

    Returns:
        str: Answer generated by RAG Pipeline.
    """
    retrieved_docs = search_similar(query, top_k=10)
    context = "\n\n".join(retrieved_docs)

    return generate_answer(context, query)

def ask(question: str):
    answer = rag_answer(question)
    #display(Markdown(answer["content"]))
    display(Markdown(render_as_markdown(answer)))

Finally we can ask a question! It will display not only the final answer but also text chunks which the LLM's answer is based on.

In [ ]:
ask("What is LLM-based Agent?")

In [ ]:
ask("What is  Reflection mechanism in AI agent?")

You can ask other questions, too. For example,

In [ ]:
ask("Is Jason Bourne the CIA agent in the movie Bourne Identity?")

## Misc.

How to check GPU memory usage:


In [ ]:
import torch
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

If GPU memory is insufficient...

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import gc

def unload_embed_model():
    global model, tokenizer
    print("Unloading embedding model...")
    del model
    del tokenizer
    model = None
    tokenizer = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared.")
    gc.collect()
    print("Models unloaded and GPU memory freed.")

def unload_chat_model():
    global chat_model, chat_tokenizer, generator
    print("Unloading chat model...")
    del chat_model
    del chat_tokenizer
    del generator
    chat_model = None
    chat_tokenizer = None
    generator = None

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared.")
    gc.collect()
    print("Models unloaded and GPU memory freed.")

def unload_models():
    """
    Unloads the embedding and chat models from GPU memory to free up resources.
    """
    global model, tokenizer, chat_model, chat_tokenizer, generator

    print("Unloading embedding model...")
    del model
    del tokenizer
    model = None
    tokenizer = None

    """
    Unloads the chat models from GPU memory to free up resources.
    """
    print("Unloading chat model...")
    del chat_model
    del chat_tokenizer
    del generator
    chat_model = None
    chat_tokenizer = None
    generator = None

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared.")
    gc.collect()
    print("Models unloaded and GPU memory freed.")

#unload_chat_model()


The example above contains a fair amount of boilerplate code. You probably do not need to write repetitive code for the same task. In the next tutorial, we will look at how to use LangChain.
